# Build a GPT from Scratch in 90 Minutes 

Welcome! In this notebook we’ll step through every component of **Mini‑GPTLate**—tokeniser, model, tracer, training—so you can _see_ and _hack_ a transformer end‑to‑end on a single CPU laptop.

---
## 0  Setup
Install the package in editable mode so any source edits are picked up instantly.

In [1]:
!pip install -q -e ..[dev]  # run once, takes ~30 s on CPU
!curl -L -o ../mini_gptlate/tokenizer.json https://huggingface.co/gpt2/raw/main/tokenizer.json 2>/dev/null || true

zsh:1: no matches found: ..[dev]


---
## 1  Tokeniser round‑trip

In [3]:
from mini_gptlate.tokeniser import get_tokeniser
tok = get_tokeniser()
s = 'Transformers are awesome!'
ids = tok.encode(s)
print('IDs:', ids)
print('Decoded →', tok.decode(ids))

KeyError: 'vocab'

---
## 2  Instantiate the model (200 lines)

In [4]:
from mini_gptlate.model import GPTLate
import torch, math
model = GPTLate()  # default 6‑layer CPU‑friendly
prompt = 'The meaning of life is'
ids = torch.tensor(tok.encode(prompt))[None]
logits, _ = model(ids)
print('Next‑token logits shape:', logits.shape)

NameError: name 'tok' is not defined

---
## 3  Live attention tracing

In [ ]:
from mini_gptlate.tracer import AttentionTracer
tracer = AttentionTracer(ids[0].tolist(), tok.decode)
# run one forward and display layer‑0 attention
_, attns = model(ids, return_attn=True)
tracer.log(0, attns[0])

---
## 4  Greedy text generation loop

In [ ]:
def generate(prompt, steps=30):
    ids = tok.encode(prompt)
    for _ in range(steps):
        inp = torch.tensor(ids[-model.config.context:])[None]
        logits, _ = model(inp)
        next_id = int(logits[0, -1].argmax())
        ids.append(next_id)
    return tok.decode(ids)

print(generate('Scientists discovered that'))

---
## 5  One‑epoch CPU training on TinyStories

In [ ]:
!pip install -q datasets  # ensures HF datasets
from mini_gptlate.train import train, argparse
args = argparse.Namespace(
    data=[], hf='tiny_stories', split='train[:1%]', context=128, batch=4, epochs=1,
    lr=3e-4, grad_acc=1, log_every=50, save_every=1e9, out='runs', ckpt='', resume=False,
    cuda=False, compile=False)
train(args)

---
## 6  LoRA fine‑tune demo (optional GPU)

In [ ]:
# Quick 200‑step adapter on TinyStories
from mini_gptlate.lora_finetune import finetune, argparse as ap2
args = ap2.Namespace(data=[], hf='tiny_stories', split='train[:1%]', context=128, batch=4,
    lr=2e-4, rank=4, steps=200, log_every=50, out='lora_demo.pt', ckpt='', cuda=False)
finetune(args)

---
## 7  Perplexity sanity check

In [ ]:
import math, torch
eval_ids = torch.tensor(tok.encode('the cat sat on the mat'))[None]
with torch.no_grad():
    logits, _ = model(eval_ids)
loss = torch.nn.functional.cross_entropy(logits.view(-1, model.config.vocab_size), eval_ids.view(-1))
print('Perplexity ≈', round(math.exp(loss.item()), 2))

---
## 🎉  Next steps
* Try `--rope` flag in model instantiation and observe attention.
* Play with `--compile` for speed gains.
* Swap tokenizer.json with your own language corpus.